<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Hunyuan3D-2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hunyuan3D-2 — Tencent's Image / Text-to-3D SuiteA complete 3D asset generation pipeline from [Tencent's Hunyuan3D-2](https://github.com/Tencent-Hunyuan/Hunyuan3D-2): given a single input image (or several views), produce a clean, watertight textured `.glb` mesh in seconds.- **Shape generation** (DiT flow-matching) — five model variants from a 0.6 B shape expert to a 1.1 B multi-view model, with and without step-distillation. Pick the model that fits your VRAM budget.- **Texture generation** (Hunyuan3D-Paint) — takes the white mesh from shape generation, plus the original input image, and paints a high-resolution diffuse texture map. Optional: requires building a small CUDA extension.- **Mesh cleanup** — FloaterRemover, DegenerateFaceRemover, FaceReducer are all wired in so the output is a usable, low-poly mesh.All weights are pulled from [tencent/Hunyuan3D-2](https://huggingface.co/tencent/Hunyuan3D-2) and [tencent/Hunyuan3D-2mini](https://huggingface.co/tencent/Hunyuan3D-2mini) on Hugging Face and cached on your Google Drive so the second-and-onward run is instant. The official [Tencent Hunyuan3D-2 Gradio app](https://huggingface.co/spaces/tencent/Hunyuan3D-2) is the basis for the UI.**Disclaimer:** The model weights are released under the [Tencent Hunyuan Community License](https://huggingface.co/tencent/Hunyuan3D-2/blob/main/LICENSE.txt) (non-commercial research use). Generated assets are not moderated by Tencent safety systems. You are responsible for downstream use.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones the [Tencent-Hunyuan/Hunyuan3D-2](https://github.com/Tencent-Hunyuan/Hunyuan3D-2) repo and
# @markdown installs `hy3dgen` plus its texture-generation CUDA extensions.
# @markdown First run: ~3-5 min. Subsequent runs: <30 s (skip when already installed).

import os, sys, subprocess, importlib, pathlib, shutil, time

REPO_DIR = '/content/Hunyuan3D-2'
REPO_URL = 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git'

# Clone the repo (idempotent, pinned to a known-good commit for stability).
PINNED_COMMIT = '40b9abf02675534b9e80e3150bd97b85c135c8c8'  # 2025-01-21 release
if not os.path.isdir(REPO_DIR):
    print(f'[Install] Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--depth=1', 'origin', PINNED_COMMIT], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', PINNED_COMMIT], check=True)
    print(f'[Install] Pinned to commit {PINNED_COMMIT[:7]}')
else:
    print(f'[Install] {REPO_DIR} already present.')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

t0 = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'ninja', 'pybind11', 'gradio==4.44.1',
                 'diffusers', 'transformers>=4.48.0', 'accelerate', 'einops', 'trimesh', 'pymeshlab',
                 'pygltflib', 'xatlas', 'rembg', 'onnxruntime', 'omegaconf', 'tqdm', 'fastapi', 'uvicorn',
                 'huggingface_hub>=0.25'], check=True)
print(f'[Install] pip install took {time.time() - t0:.1f}s')

# Build the custom CUDA extensions needed for texture generation. These are NOT required
# for shape generation, so we wrap the build in a try/except and only fail at the Paint
# tab if it really doesn't work.
def _try_build_cuda_extensions():
    from torch.utils.cpp_extension import CUDA_HOME
    if not CUDA_HOME or not os.path.exists(os.path.join(CUDA_HOME, 'bin', 'nvcc')):
        return 'no-nvcc', 'No nvcc found; texture synthesis will be disabled.'
    log = []
    # 1) custom_rasterizer (needed by texture pipeline)
    t0 = time.time()
    try:
        subprocess.run([sys.executable, 'setup.py', 'install'], cwd='hy3dgen/texgen/custom_rasterizer',
                       check=True, capture_output=True)
        log.append(f'custom_rasterizer OK ({time.time() - t0:.0f}s)')
    except subprocess.CalledProcessError as e:
        return 'rasterizer-failed', f'custom_rasterizer build failed: {e.stderr.decode()[-500:]}'
    # 2) mesh_processor (C++ only, used by the mesh cleanup utilities)
    t0 = time.time()
    try:
        subprocess.run([sys.executable, 'setup.py', 'install'],
                       cwd='hy3dgen/texgen/differentiable_renderer', check=True, capture_output=True)
        log.append(f'mesh_processor OK ({time.time() - t0:.0f}s)')
    except subprocess.CalledProcessError as e:
        log.append(f'mesh_processor warning: {e.stderr.decode()[-200:] if e.stderr else e}')
    return 'ok', ' ; '.join(log)

print('[Install] Building CUDA extensions (may take a couple of minutes) ...')
rasterizer_status, rasterizer_msg = _try_build_cuda_extensions()
print(f'[Install] CUDA build: {rasterizer_status} — {rasterizer_msg}')

# Sanity-check imports.
try:
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline, FaceReducer, FloaterRemover, DegenerateFaceRemover
    from hy3dgen.rembg import BackgroundRemover
    print('[Install] hy3dgen.shapegen OK')
except Exception as e:
    raise SystemExit(f'hy3dgen import failed: {e}')

HAS_TEXTUREGEN = rasterizer_status == 'ok'
if not HAS_TEXTUREGEN:
    print('[Install] Texture synthesis will be disabled in the UI.')

print(f'[Install] DONE in {time.time() - t0:.1f}s total.')
print(f'[Install] GPU: {__import__("torch").cuda.get_device_name(0)}')

In [ ]:
# @title STEP 2 — Cache Weights & Example Images
# @markdown Downloads the DiT + Paint weights into a Google Drive cache (≈ 4-12 GB depending on
# @markdown how many sub-models you select) and fetches 6 example images from the upstream repo.

import os, sys, time, shutil, pathlib, requests
from huggingface_hub import snapshot_download, hf_hub_download
from tqdm.notebook import tqdm

CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache/huggingface/hub')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT.parent)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT)
print(f'[Cache] HF cache root: {CACHE_ROOT}')

ASSETS_DIR = pathlib.Path('/content/hy3d_assets')
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
EXAMPLE_IMAGES = ['052.png', '073.png', '1008.png', '1022.png', '1146.png', '1180.png']

# Model registry: each entry is (repo_id, [subfolders to download]).
MODEL_REGISTRY = {
    '2mini':      ('tencent/Hunyuan3D-2mini',
                   ['hunyuan3d-dit-v2-mini', 'hunyuan3d-dit-v2-mini-turbo']),
    '2mv-turbo':  ('tencent/Hunyuan3D-2mv',
                   ['hunyuan3d-dit-v2-mv', 'hunyuan3d-dit-v2-mv-turbo']),
    '2':          ('tencent/Hunyuan3D-2',
                   ['hunyuan3d-dit-v2-0', 'hunyuan3d-dit-v2-0-turbo', 'hunyuan3d-dit-v2-0-fast',
                    'hunyuan3d-paint-v2-0', 'hunyuan3d-paint-v2-0-turbo', 'hunyuan3d-delight-v2-0']),
}
DEFAULT_VARIANT = '2mini-turbo'  # smallest, fastest, fits T4 comfortably

def _human(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def _dir_size(p):
    p = pathlib.Path(p)
    if not p.exists(): return 0
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())

def download_variant(variant_key, force=False):
    if variant_key not in MODEL_REGISTRY:
        raise ValueError(f'unknown variant: {variant_key}; choose from {list(MODEL_REGISTRY)}')
    repo_id, subfolders = MODEL_REGISTRY[variant_key]
    local_dir = CACHE_ROOT / repo_id.replace('/', '_')
    existing = _dir_size(local_dir)
    if existing > 200_000_000 and not force:
        print(f'[Cache] {variant_key} already cached ({_human(existing)}). Skipping.')
        return str(local_dir)
    print(f'[Cache] Downloading {variant_key} from {repo_id} (subfolders: {subfolders}) ...')
    snapshot_download(
        repo_id=repo_id,
        allow_patterns=[f'{sf}/*' for sf in subfolders] + ['*.json', '*.txt'],
        local_dir=str(local_dir),
        local_dir_use_symlinks=False,
        tqdm_class=tqdm,
    )
    print(f'[Cache] {variant_key} now on disk: {_human(_dir_size(local_dir))}')
    return str(local_dir)

# -- Fetch example images ----------------------------------------
def fetch_examples(force=False):
    got = []
    for name in EXAMPLE_IMAGES:
        dst = ASSETS_DIR / name
        if dst.exists() and dst.stat().st_size > 1000 and not force:
            got.append(str(dst)); continue
        url = f'https://raw.githubusercontent.com/Tencent-Hunyuan/Hunyuan3D-2/main/assets/example_images/{name}'
        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            dst.write_bytes(r.content)
            got.append(str(dst))
        except Exception as e:
            print(f'  [Example] {name}: {e}')
    print(f'[Cache] {len(got)}/{len(EXAMPLE_IMAGES)} example images in {ASSETS_DIR}')
    return got

# Interactive chooser: keep the default small set; user can edit before running.
print('[Cache] Default variant: 2mini-turbo (others: 2-turbo, 2, 2mv-turbo, 2mini)')
print('[Cache] Edit VARIANTS below to download more.')
VARIANTS = ['2mini-turbo']  # add '2-turbo' and '2' to also get the full model + Paint

if shutil.disk_usage('/content').free < 25 * 1024**3:
    print('[Cache] WARNING: less than 25 GB free on /content. Bigger variants may not fit.')

_dirs = {}
for v in VARIANTS:
    _dirs[v] = download_variant(v)

examples = fetch_examples()
print(f'\n[Cache] DONE. Models in: {list(_dirs.values())}')
print(f'[Cache] Examples in: {ASSETS_DIR}')
print('Run Step 3 next.')

In [ ]:
# @title STEP 3 — Pipeline Wrappers (Lazy Load)
# @markdown Defines lazy-loading wrappers for the shape and texture pipelines so the first
# @markdown model variant you pick is the one that gets loaded into VRAM.

import gc, os, sys, time, traceback, pathlib, json, random
import torch, numpy as np, trimesh
from PIL import Image

REPO_DIR = '/content/Hunyuan3D-2'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from hy3dgen.shapegen import (Hunyuan3DDiTFlowMatchingPipeline, FaceReducer, FloaterRemover,
                             DegenerateFaceRemover)
from hy3dgen.shapegen.pipelines import export_to_trimesh
from hy3dgen.rembg import BackgroundRemover

try:
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    _PAINT_IMPORT_OK = True
except Exception as _e:
    print(f'[Core] Paint import failed: {_e}')
    Hunyuan3DPaintPipeline = None
    _PAINT_IMPORT_OK = False

VARIANT_TO_REPO = {
    '2mini-turbo': ('tencent/Hunyuan3D-2mini', 'hunyuan3d-dit-v2-mini-turbo'),
    '2mini':       ('tencent/Hunyuan3D-2mini', 'hunyuan3d-dit-v2-mini'),
    '2-turbo':     ('tencent/Hunyuan3D-2',     'hunyuan3d-dit-v2-0-turbo'),
    '2-fast':      ('tencent/Hunyuan3D-2',     'hunyuan3d-dit-v2-0-fast'),
    '2':           ('tencent/Hunyuan3D-2',     'hunyuan3d-dit-v2-0'),
    '2mv-turbo':   ('tencent/Hunyuan3D-2mv',   'hunyuan3d-dit-v2-mv-turbo'),
    '2mv':         ('tencent/Hunyuan3D-2mv',   'hunyuan3d-dit-v2-mv'),
}

class ShapeEngine:
    """Holds a Hunyuan3DDiTFlowMatchingPipeline + cleanup workers, swap-friendly."""
    def __init__(self):
        self.pipeline = None
        self.variant = None
        self.floater = FloaterRemover()
        self.degenerate = DegenerateFaceRemover()
        self.reducer = FaceReducer()
        self.rmbg = BackgroundRemover()

    def load(self, variant):
        if self.pipeline is not None and self.variant == variant:
            print(f'[Shape] {variant} already loaded.')
            return
        self.unload()
        if variant not in VARIANT_TO_REPO:
            raise ValueError(f'unknown variant: {variant}')
        repo_id, sub = VARIANT_TO_REPO[variant]
        print(f'[Shape] Loading {repo_id} / {sub} ...')
        t0 = time.time()
        self.pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            repo_id, subfolder=sub, use_safetensors=True, device='cuda',
        )
        self.variant = variant
        print(f'[Shape] Loaded in {time.time() - t0:.1f}s. VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

    def unload(self):
        if self.pipeline is not None:
            print(f'[Shape] Unloading {self.variant} ...')
            del self.pipeline
            self.pipeline = None
            self.variant = None
            gc.collect(); torch.cuda.empty_cache()

    def is_mv(self):
        return self.variant is not None and 'mv' in self.variant

    def generate(self, image_pil, num_inference_steps, guidance_scale, seed, octree_resolution, num_chunks):
        if self.pipeline is None:
            raise RuntimeError('shape engine not loaded')
        g = torch.Generator().manual_seed(int(seed))
        out = self.pipeline(
            image=image_pil,
            num_inference_steps=int(num_inference_steps),
            guidance_scale=float(guidance_scale),
            generator=g,
            octree_resolution=int(octree_resolution),
            num_chunks=int(num_chunks),
            output_type='mesh',
        )
        mesh = export_to_trimesh(out)[0]
        return mesh

    def cleanup(self, mesh):
        mesh = self.floater(mesh)
        mesh = self.degenerate(mesh)
        mesh = self.reducer(mesh)
        return mesh

class PaintEngine:
    def __init__(self):
        self.pipeline = None
        if not _PAINT_IMPORT_OK:
            self.disabled_reason = 'Paint import failed (likely missing custom_rasterizer).'
        else:
            self.disabled_reason = None

    def load(self):
        if not _PAINT_IMPORT_OK:
            raise RuntimeError(self.disabled_reason)
        if self.pipeline is not None:
            print('[Paint] already loaded.'); return
        print('[Paint] Loading tencent/Hunyuan3D-2 (Hunyuan3D-Paint-v2-0) ...')
        t0 = time.time()
        self.pipeline = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
        self.pipeline.enable_model_cpu_offload()
        print(f'[Paint] Loaded in {time.time() - t0:.1f}s.')

    def unload(self):
        if self.pipeline is not None:
            del self.pipeline; self.pipeline = None
            gc.collect(); torch.cuda.empty_cache()

    def texture(self, mesh, image):
        if self.pipeline is None:
            raise RuntimeError('paint engine not loaded')
        return self.pipeline(mesh, image=image)

SHAPE = ShapeEngine()
PAINT = PaintEngine()

def _strip_bg(image, do_rembg):
    if image is None: return None
    if image.mode != 'RGBA': image = image.convert('RGBA')
    if do_rembg or image.mode == 'RGB':
        return SHAPE.rmbg(image.convert('RGB'))
    return image

def gen_shape(image, steps, cfg, seed, octree, num_chunks, do_rembg=True):
    if image is None:
        raise ValueError('image is required')
    image = _strip_bg(image, do_rembg)
    mesh = SHAPE.generate(image, steps, cfg, seed, octree, num_chunks)
    mesh = SHAPE.cleanup(mesh)
    return mesh, image

def gen_textured(mesh, image):
    if not _PAINT_IMPORT_OK:
        raise RuntimeError('Paint pipeline not available (CUDA extension missing).')
    return PAINT.texture(mesh, image)

print('[Core] Engines ready.')
print(f'[Core]   Paint available: {_PAINT_IMPORT_OK}')
print('[Core]   Run Step 4 to launch the UI.')

In [ ]:
# @title STEP 4 — Launch Gradio UI
# @markdown Opens a Gradio app with one tab per model variant + a Paint tab + a VRAM tab.

import os, sys, time, json, uuid, shutil, pathlib, gc, traceback
import torch, numpy as np, trimesh, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, HTML

REPO_DIR = '/content/Hunyuan3D-2'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

SAVE_DIR = pathlib.Path('/content/hy3d_runs')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR = pathlib.Path('/content/hy3d_assets')
EXAMPLE_PATHS = sorted(str(p) for p in ASSETS_DIR.glob('*.png')) if ASSETS_DIR.exists() else []

def _new_run_dir():
    p = SAVE_DIR / f'run_{int(time.time())}_{uuid.uuid4().hex[:6]}'
    p.mkdir(parents=True, exist_ok=True)
    return p

def _save_mesh(mesh, folder, name='white_mesh.glb', textured=False):
    p = folder / name
    mesh.export(str(p), include_normals=textured)
    return str(p)

def _model_viewer_html(folder, height=620, textured=False):
    fname = 'textured_mesh.glb' if textured else 'white_mesh.glb'
    rel = f'./{fname}'
    return f"""
    <div style='height: {height}px; width: 100%; border-radius: 8px; border: 1px solid #e5e7eb;'>
      <model-viewer src="{rel}/" alt="3D mesh" auto-rotate camera-controls
        style='height: {height-2}px; width: 100%;'
        shadow-intensity='0.9' environment-image='neutral'></model-viewer>
    </div>
    <script type='module' src='https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js'></script>
    """

def do_generate_shape(variant, image, steps, cfg, seed, octree, num_chunks,
                      do_rembg, randomize_seed, progress=gr.Progress()):
    try:
        SHAPE.load(variant)
        if randomize_seed:
            seed = random.randint(0, 1_000_000)
        progress(0.05, desc=f'Running {variant} ...')
        t0 = time.time()
        mesh, proc_img = gen_shape(image, int(steps), float(cfg), int(seed), int(octree), int(num_chunks), bool(do_rembg))
        progress(0.85, desc='Saving mesh ...')
        folder = _new_run_dir()
        path = _save_mesh(mesh, folder)
        html = _model_viewer_html(folder)
        stats = {'variant': variant, 'seed': int(seed), 'faces': int(mesh.faces.shape[0]),
                 'vertices': int(mesh.vertices.shape[0]), 'elapsed_s': round(time.time() - t0, 1)}
        return html, path, proc_img, stats, int(seed)
    except Exception as e:
        traceback.print_exc()
        return None, None, image, {'error': str(e)}, int(seed)

def do_generate_textured(variant, image, steps, cfg, seed, octree, num_chunks,
                         do_rembg, randomize_seed, progress=gr.Progress()):
    try:
        SHAPE.load(variant)
        if randomize_seed:
            seed = random.randint(0, 1_000_000)
        progress(0.05, desc='Shape ...')
        t0 = time.time()
        mesh, proc_img = gen_shape(image, int(steps), float(cfg), int(seed), int(octree), int(num_chunks), bool(do_rembg))
        progress(0.55, desc='Loading Paint ...')
        PAINT.load()
        progress(0.65, desc='Painting texture ...')
        textured = gen_textured(mesh, proc_img)
        progress(0.92, desc='Saving ...')
        folder = _new_run_dir()
        path_white = _save_mesh(mesh, folder, 'white_mesh.glb')
        path_tex = _save_mesh(textured, folder, 'textured_mesh.glb', textured=True)
        html = _model_viewer_html(folder, textured=False)
        html_tex = _model_viewer_html(folder, textured=True)
        stats = {'variant': variant, 'seed': int(seed),
                 'faces_white': int(mesh.faces.shape[0]),
                 'faces_textured': int(textured.faces.shape[0]),
                 'elapsed_s': round(time.time() - t0, 1)}
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return html, html_tex, path_white, path_tex, proc_img, stats, int(seed)
    except Exception as e:
        traceback.print_exc()
        return None, None, None, None, image, {'error': str(e)}, int(seed)

def do_texture_existing(mesh_file, image, progress=gr.Progress()):
    try:
        if mesh_file is None:
            raise ValueError('Please upload a .glb / .obj / .ply mesh.')
        if image is None:
            raise ValueError('Please provide the image to paint from.')
        PAINT.load()
        progress(0.2, desc='Loading mesh ...')
        mesh = trimesh.load(mesh_file, force='mesh')
        progress(0.4, desc='Painting ...')
        textured = PAINT.texture(mesh, image)
        progress(0.9, desc='Saving ...')
        folder = _new_run_dir()
        path_in = _save_mesh(mesh, folder, 'input_mesh.glb')
        path_tex = _save_mesh(textured, folder, 'textured_mesh.glb', textured=True)
        html = _model_viewer_html(folder, textured=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return html, path_in, path_tex
    except Exception as e:
        traceback.print_exc()
        return None, None, None

def vram_free_shape():
    SHAPE.unload()
    return f'[VRAM] Shape engine freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated.'

def vram_free_paint():
    PAINT.unload()
    return f'[VRAM] Paint engine freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated.'

def vram_free_all():
    SHAPE.unload(); PAINT.unload(); gc.collect(); torch.cuda.empty_cache()
    return f'[VRAM] All engines freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated.'

def vram_status():
    if not torch.cuda.is_available(): return 'No GPU.'
    a = torch.cuda.memory_allocated() / 1024**3
    r = torch.cuda.memory_reserved() / 1024**3
    t = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return f'Allocated: {a:.1f} GB\nReserved:  {r:.1f} GB\nTotal:     {t:.1f} GB\nShape loaded: {SHAPE.variant or "—"}\nPaint loaded: {PAINT.pipeline is not None}'

VARIANT_CHOICES = list(VARIANT_TO_REPO.keys())
DEFAULT_VARIANT = '2mini-turbo'

with gr.Blocks(title='Hunyuan3D-2 Colab', theme=gr.themes.Base(), analytics_enabled=False) as demo:
    gr.HTML("""
    <div style='font-size: 1.8em; font-weight: bold; text-align: center; margin: 4px 0;'>
      Hunyuan3D-2 — Image / Multi-View → Textured 3D</div>
    <div style='text-align: center; color: #6b7280;'>
      Pick a model variant in the sidebar, then drop in an image (or use the Examples gallery).</div>
    """)
    with gr.Tabs():
        # -- Gen shape tab -------------------------------------------
        with gr.Tab('Shape (white mesh)'):
            with gr.Row():
                with gr.Column(scale=3):
                    variant = gr.Dropdown(label='Model variant', choices=VARIANT_CHOICES,
                                           value=DEFAULT_VARIANT,
                                           info='2mini-turbo is fastest. 2 = highest quality. 2mv-* = multi-view.')
                    image_in = gr.Image(label='Input image', type='pil', image_mode='RGBA', height=280)
                    do_rembg = gr.Checkbox(value=True, label='Remove background',
                                           info='Recommended unless your image already has a clean alpha channel.')
                    with gr.Accordion('Advanced options', open=False):
                        steps = gr.Slider(1, 100, value=30, step=1, label='Inference steps',
                                          info='5 for turbo variants, 30 for full models.')
                        cfg = gr.Slider(1.0, 15.0, value=5.5, step=0.1, label='Guidance scale')
                        octree = gr.Slider(64, 512, value=256, step=32, label='Octree resolution',
                                           info='Higher = more detail, more VRAM.')
                        num_chunks = gr.Slider(1000, 200_000, value=8000, step=1000, label='Decode chunks',
                                               info='Lower if you hit VRAM OOM during decoding.')
                        seed = gr.Slider(0, 1_000_000, value=1234, step=1, label='Seed')
                        randomize_seed = gr.Checkbox(value=False, label='Randomize seed')
                    btn_shape = gr.Button('Generate Shape', variant='primary')
                with gr.Column(scale=4):
                    out_html = gr.HTML("""<div style='height: 560px; width: 100%; border-radius: 8px;
                        border: 1px solid #e5e7eb; display: flex; align-items: center; justify-content: center;
                        color: #8d8d8d;'>No mesh yet.</div>""")
                    with gr.Row():
                        out_file = gr.File(label='Download .glb')
                        proc_image = gr.Image(label='Processed input', type='pil', height=120, interactive=False)
                    out_stats = gr.JSON(label='Run stats')
                with gr.Column(scale=2):
                    if EXAMPLE_PATHS:
                        gr.Examples(examples=[[p] for p in EXAMPLE_PATHS], inputs=[image_in],
                                    label='Example images', examples_per_page=6)
            btn_shape.click(
                do_generate_shape,
                inputs=[variant, image_in, steps, cfg, seed, octree, num_chunks, do_rembg, randomize_seed],
                outputs=[out_html, out_file, proc_image, out_stats, seed],
            )
        # -- Gen textured tab ----------------------------------------
        with gr.Tab('Shape + Texture'):
            gr.HTML("""<div style='background: #fef3c7; color: #92400e; padding: 8px 12px; border-radius: 6px;
                margin: 6px 0;'>Texture generation is slower and uses more VRAM. It will load the Paint
                engine after the shape is built. <b>Recommended GPU: L4 (24 GB) or A100.</b></div>""")
            with gr.Row():
                with gr.Column(scale=3):
                    variant_t = gr.Dropdown(label='Model variant', choices=VARIANT_CHOICES,
                                            value=DEFAULT_VARIANT)
                    image_t = gr.Image(label='Input image', type='pil', image_mode='RGBA', height=240)
                    do_rembg_t = gr.Checkbox(value=True, label='Remove background')
                    with gr.Accordion('Advanced options', open=False):
                        steps_t = gr.Slider(1, 100, value=30, step=1, label='Inference steps')
                        cfg_t = gr.Slider(1.0, 15.0, value=5.5, step=0.1, label='Guidance scale')
                        octree_t = gr.Slider(64, 512, value=256, step=32, label='Octree resolution')
                        num_chunks_t = gr.Slider(1000, 200_000, value=8000, step=1000, label='Decode chunks')
                        seed_t = gr.Slider(0, 1_000_000, value=1234, step=1, label='Seed')
                        randomize_seed_t = gr.Checkbox(value=False, label='Randomize seed')
                    btn_tex = gr.Button('Generate Shape + Texture', variant='primary')
                with gr.Column(scale=4):
                    with gr.Tabs():
                        with gr.Tab('White mesh'):
                            out_white_html = gr.HTML("""<div style='height: 480px; width: 100%;
                                border: 1px solid #e5e7eb; display: flex; align-items: center;
                                justify-content: center; color: #8d8d8d;'>No mesh yet.</div>""")
                        with gr.Tab('Textured mesh'):
                            out_tex_html = gr.HTML("""<div style='height: 480px; width: 100%;
                                border: 1px solid #e5e7eb; display: flex; align-items: center;
                                justify-content: center; color: #8d8d8d;'>No mesh yet.</div>""")
                    with gr.Row():
                        out_white_file = gr.File(label='White .glb')
                        out_tex_file = gr.File(label='Textured .glb')
                        proc_image_t = gr.Image(label='Processed input', type='pil', height=120, interactive=False)
                    out_stats_t = gr.JSON(label='Run stats')
                with gr.Column(scale=2):
                    if EXAMPLE_PATHS:
                        gr.Examples(examples=[[p] for p in EXAMPLE_PATHS], inputs=[image_t],
                                    label='Example images', examples_per_page=6)
            btn_tex.click(
                do_generate_textured,
                inputs=[variant_t, image_t, steps_t, cfg_t, seed_t, octree_t, num_chunks_t,
                        do_rembg_t, randomize_seed_t],
                outputs=[out_white_html, out_tex_html, out_white_file, out_tex_file, proc_image_t,
                         out_stats_t, seed_t],
            )
        # -- Texture existing tab -----------------------------------
        with gr.Tab('Texture an existing mesh'):
            gr.HTML("""<div style='background: #f0fdf4; color: #166534; padding: 8px 12px; border-radius: 6px;
                margin: 6px 0;'>Upload a hand-made <code>.glb / .obj / .ply</code> mesh plus an image to
                paint from. Useful for re-texturing outputs from Cube 3D, CubePart, or anything else.</div>""")
            with gr.Row():
                with gr.Column(scale=3):
                    mesh_in = gr.File(label='Input mesh (.glb / .obj / .ply)', file_types=['.glb', '.obj', '.ply', '.stl'])
                    image_e = gr.Image(label='Image to paint from', type='pil', image_mode='RGBA', height=240)
                    btn_e = gr.Button('Paint Texture', variant='primary')
                with gr.Column(scale=5):
                    out_e_html = gr.HTML("""<div style='height: 520px; width: 100%; border: 1px solid #e5e7eb;
                        display: flex; align-items: center; justify-content: center; color: #8d8d8d;'>
                        No mesh yet.</div>""")
                    with gr.Row():
                        out_e_input = gr.File(label='Input mesh (saved)')
                        out_e_tex = gr.File(label='Textured .glb')
            btn_e.click(do_texture_existing, inputs=[mesh_in, image_e],
                        outputs=[out_e_html, out_e_input, out_e_tex])
        # -- VRAM tab -----------------------------------------------
        with gr.Tab('VRAM'):
            gr.HTML("""<div>Use these buttons to swap pipelines in and out of VRAM. 
                T4 (16 GB) can only hold one of the larger pipelines at a time.</div>""")
            vram_text = gr.Textbox(label='Status', value=vram_status, interactive=False, lines=6)
            with gr.Row():
                free_shape_btn = gr.Button('Free shape engine')
                free_paint_btn = gr.Button('Free paint engine')
                free_all_btn = gr.Button('Free everything', variant='stop')
                refresh_btn = gr.Button('Refresh status')
            free_shape_btn.click(vram_free_shape, outputs=vram_text)
            free_paint_btn.click(vram_free_paint, outputs=vram_text)
            free_all_btn.click(vram_free_all, outputs=vram_text)
            refresh_btn.click(vram_status, outputs=vram_text)

clear_output()
print('[UI] Launching ...')
demo.queue(max_size=4, default_concurrency_limit=2)
demo.load(lambda: gr.Info("Welcome to Hunyuan3D-2. Pick a model variant in the Shape tab, drop an image, click Generate."), None)
demo.launch(share=True, show_error=True, height=900, prevent_thread_lock=False)
print('\n[UI] Public URL above. Open it in a new tab.')
print('[UI] The shape variant is loaded on first click in the Shape tab.')

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings a tiny URL every 60 s to keep the Colab tab active for the 90-min idle limit.

import time, requests, datetime, threading, IPython
_stop = threading.Event()
def _pinger():
    while not _stop.is_set():
        try:
            requests.get('https://huggingface.co/api/models/tencent/Hunyuan3D-2', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} OK — UI still running above. Stop with `_stop.set()`.')
        except Exception as e:
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} WARN: {e}')
        _stop.wait(60)
_t = threading.Thread(target=_pinger, daemon=True)
_t.start()
print('[Keep-Alive] Started. Safe to keep this cell running.')

In [ ]:
# @title STEP 6 — Quick Test (single inference smoke test)
# @markdown Runs a single end-to-end shape generation with the 2mini-turbo model and a downloaded example
# @markdown image. Confirms that Step 1 + 2 + 3 are all wired correctly without launching the UI.

import os, sys, time, pathlib
import torch, trimesh
from PIL import Image
from IPython.display import FileLink, display

REPO_DIR = '/content/Hunyuan3D-2'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

QUICK_VARIANT = '2mini-turbo'
QUICK_STEPS = 5
QUICK_OCTREE = 196
QUICK_SEED = 0
QUICK_DO_PAINT = False  # set True to also exercise the texture pipeline

ASSETS_DIR = pathlib.Path('/content/hy3d_assets')
examples = sorted(ASSETS_DIR.glob('*.png')) if ASSETS_DIR.exists() else []
if not examples:
    raise SystemExit('No example images found. Re-run Step 2 to download them.')
img = Image.open(examples[0]).convert('RGBA')
print(f'[QuickTest] Using {examples[0].name} ({img.size})')

SHAPE.load(QUICK_VARIANT)
t0 = time.time()
mesh, proc = gen_shape(img, QUICK_STEPS, 5.5, QUICK_SEED, QUICK_OCTREE, 8000, do_rembg=True)
print(f'[QuickTest] shape generation: {time.time() - t0:.1f}s, {mesh.faces.shape[0]} faces')

out_dir = pathlib.Path('/content/hy3d_runs/quicktest')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'quick_mesh.glb'
mesh.export(str(out_path), include_normals=False)
print(f'[QuickTest] saved {out_path}')
display(FileLink(str(out_path)))

if QUICK_DO_PAINT and _PAINT_IMPORT_OK:
    print('[QuickTest] also running texture ...')
    PAINT.load()
    t0 = time.time()
    textured = gen_textured(mesh, proc)
    print(f'[QuickTest] texture: {time.time() - t0:.1f}s')
    tex_path = out_dir / 'quick_mesh_textured.glb'
    textured.export(str(tex_path), include_normals=True)
    print(f'[QuickTest] saved {tex_path}')
    display(FileLink(str(tex_path)))
elif QUICK_DO_PAINT:
    print('[QuickTest] QUICK_DO_PAINT=True but Paint is not available (custom_rasterizer missing).')

print(f'\n[QuickTest] DONE. Final VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

In [ ]:
# @title STEP 7 — Batch Generation from a .txt list
# @markdown Reads a text file of image paths (one per line) and generates meshes for each.
# @markdown Saves them to a per-run folder. Use this for sweeping a folder of reference images.

import os, sys, time, pathlib, traceback
import torch, trimesh
from PIL import Image

REPO_DIR = '/content/Hunyuan3D-2'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# --- Edit these before running ---------------------------------
LIST_FILE = '/content/hy3d_assets/batch_list.txt'  # one image path per line
BATCH_VARIANT = '2mini-turbo'
BATCH_STEPS = 5
BATCH_CFG = 5.5
BATCH_OCTREE = 196
BATCH_NUM_CHUNKS = 8000
BATCH_SEED = 0
BATCH_DO_REMBG = True
BATCH_DO_PAINT = False
OUT_ROOT = pathlib.Path('/content/hy3d_runs/batch')
MAX_ITEMS = 0  # 0 = no cap
# --------------------------------------------------------------

list_path = pathlib.Path(LIST_FILE)
if not list_path.exists():
    # auto-bootstrap: use the example images we already downloaded.
    examples = sorted(pathlib.Path('/content/hy3d_assets').glob('*.png'))
    list_path.write_text('\n'.join(str(p) for p in examples) + '\n')
    print(f'[Batch] No list file found, bootstrapped {len(examples)} examples into {list_path}')

lines = [l.strip() for l in list_path.read_text().splitlines() if l.strip() and not l.startswith('#')]
if MAX_ITEMS: lines = lines[:MAX_ITEMS]
print(f'[Batch] {len(lines)} image(s) queued. Variant: {BATCH_VARIANT}')

SHAPE.load(BATCH_VARIANT)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
batch_dir = OUT_ROOT / f'batch_{int(time.time())}'
batch_dir.mkdir(parents=True, exist_ok=True)

ok = 0; fail = 0
for i, line in enumerate(lines, 1):
    try:
        p = pathlib.Path(line)
        if not p.exists():
            print(f'  [{i:03d}] SKIP (not found): {line}'); fail += 1; continue
        img = Image.open(p).convert('RGBA')
        t0 = time.time()
        mesh, proc = gen_shape(img, BATCH_STEPS, BATCH_CFG, BATCH_SEED, BATCH_OCTREE, BATCH_NUM_CHUNKS, BATCH_DO_REMBG)
        mesh = SHAPE.cleanup(mesh)
        safe = ''.join(c if c.isalnum() else '_' for c in p.stem[:40]).strip('_') or f'item_{i:02d}'
        out_path = batch_dir / f'{i:03d}_{safe}.glb'
        mesh.export(str(out_path), include_normals=BATCH_DO_PAINT)
        elapsed = time.time() - t0
        if BATCH_DO_PAINT and _PAINT_IMPORT_OK:
            try:
                PAINT.load()
                textured = gen_textured(mesh, proc)
                tex_path = batch_dir / f'{i:03d}_{safe}_textured.glb'
                textured.export(str(tex_path), include_normals=True)
                print(f'  [{i:03d}] {p.name} -> {out_path.name} + {tex_path.name} ({elapsed:.1f}s, shape)')
            except Exception as e:
                print(f'  [{i:03d}] {p.name} -> {out_path.name} (shape OK; paint failed: {e})')
        else:
            print(f'  [{i:03d}] {p.name} -> {out_path.name} ({elapsed:.1f}s)')
        ok += 1
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        print(f'  [{i:03d}] EXCEPTION on {line}: {e}'); traceback.print_exc(); fail += 1

print(f'\n[Batch] DONE: {ok} OK / {fail} failed / {len(lines)} total -> {batch_dir}')